In [2]:
pip install transformers torch --break-system-packages

Note: you may need to restart the kernel to use updated packages.


In [1]:
pip install transformers torch

Note: you may need to restart the kernel to use updated packages.


In [3]:
from transformers import pipeline
import warnings
warnings.filterwarnings("ignore")
#from transformers import pipeline
# Load a small open-source text-generation model
generator = pipeline("text-generation", model="gpt2")

prompt = "Artificial Intelligence will change education by"
#prompt = "I can't"
output = generator(
    prompt,
    max_new_tokens=5,
    num_return_sequences=1,
    pad_token_id=generator.tokenizer.eos_token_id  # silences the pad_token warning
)
print(output[0]["generated_text"])

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'pad_token_id', 'max_new_tokens', 'num_return_sequences'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=5) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Artificial Intelligence will change education by building new skills and technology


In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokens = tokenizer.tokenize("Generative AI very usefull to create any new music")
print(tokens)
print("Token count:", len(tokens))

['Gener', 'ative', 'ĠAI', 'Ġvery', 'Ġuse', 'full', 'Ġto', 'Ġcreate', 'Ġany', 'Ġnew', 'Ġmusic']
Token count: 11


In [5]:
from transformers import pipeline

# Load a small open-source text-generation model
# it's a base language model trained purely to predict the next word from a huge text corpus
generator = pipeline("text-generation", model="gpt2")

#prompt = "Artificial Intelligence will change education by"
prompt = "I can't"
output = generator(
    prompt,
    max_new_tokens=3,
    num_return_sequences=1,
    pad_token_id=generator.tokenizer.eos_token_id  # silences the pad_token warning
)

print(output[0]["generated_text"])

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=3) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


I can't even explain why


In [6]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis")

result=classifier("I absolutely love to learning about AI")
print(result)

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9997941851615906}]


In [7]:
from transformers import pipeline

# Instead of the default, we are explicitly calling a specialized model 
# 'cardiffnlp/twitter-roberta-base-sentiment-latest' (trained heavily on social media text)
specific_classifier = pipeline(
    "sentiment-analysis", 
    model="cardiffnlp/twitter-roberta-base-sentiment-latest"
)

#result = specific_classifier("This new update is fire! 🔥")
result = specific_classifier("I rarely like to learn AI")
print(result)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'label': 'negative', 'score': 0.7106242179870605}]


In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"

# STEP 1: Load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

#text = "This new update is fire! 🔥"
text = "I rarely like to learn AI"
# STEP 2: Tokenize the input text
# 'return_tensors="pt"' tells it to return PyTorch tensors
inputs = tokenizer(text, return_tensors="pt")

# STEP 3: Run the text through the model (Inference)
with torch.no_grad():  # Disables gradient calculation for faster inference since we aren't training
    outputs = model(**inputs)

# STEP 4: Process the raw outputs (logits)
# The model outputs raw numbers called 'logits' for each class
logits = outputs.logits

# Apply Softmax to convert raw logits into actual percentage probabilities (0 to 1)
probabilities = torch.softmax(logits, dim=-1).tolist()[0]

# STEP 5: Map probabilities to the model's text labels
# This model uses labels: 0 -> Negative, 1 -> Neutral, 2 -> Positive
labels = model.config.id2label

# Find the highest probability index
highest_prob_idx = torch.argmax(logits, dim=-1).item()

# Print the final result
print(f"Label: {labels[highest_prob_idx]}")
print(f"Score: {probabilities[highest_prob_idx]:.4f}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Label: negative
Score: 0.7106
